### Import Statements

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

### Read in CSV

In [2]:
df = pd.read_csv("play_by_play_2025.csv", low_memory = False)

### Data Cleaning and Aggregation

In [3]:
# Filter only by Regular Season data and denote it as our training data (done so that if the Notebook is ran again in the future, we are not including Postseason data)
train = df[df.season_type == "REG"]
train.season_type.unique()

array(['REG'], dtype=object)

In [4]:
train

,play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,out_of_bounds,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe
0,1,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,NaN,NaN,NaN,...,0,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,40,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,0,0,-0.352700,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,63,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,0,0,-0.190052,NaN,NaN,NaN,NaN,NaN,0.511128,-51.112807
3,85,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,1,0,1.317340,0.939998,4.750889,3.0,0.666726,0.43911,0.668940,33.105969
4,115,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,0,0,-1.694360,NaN,NaN,NaN,NaN,NaN,0.492038,50.796208
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46447,4120,2025_18_WAS_PHI,2026010412,PHI,WAS,REG,18,PHI,home,WAS,...,0,1,-0.137928,NaN,NaN,NaN,NaN,NaN,0.994123,0.587696
46448,4143,2025_18_WAS_PHI,2026010412,PHI,WAS,REG,18,PHI,home,WAS,...,0,1,-0.133620,NaN,NaN,NaN,NaN,NaN,0.983971,1.602894
46449,4163,2025_18_WAS_PHI,2026010412,PHI,WAS,REG,18,PHI,home,WAS,...,0,1,-2.038239,0.954084,9.474893,4.0,1.000000,1.00000,0.995855,0.414509
46450,4186,2025_18_WAS_PHI,2026010412,PHI,WAS,REG,18,WAS,away,PHI,...,0,1,-2.090118,NaN,NaN,NaN,NaN,NaN,NaN,NaN


##### Grouping By Team/Offensive/Defensive Statistics

- Approach: Divide Statistics into Per Game, Per Drive, and Per Play Frequencies depending on the feature

Team Statistics

In [5]:
# Win Rate

# Home Wins for Each Team
home_wins = train[train['home_score'] > train['away_score']].groupby("home_team")['game_id'].nunique()
home_wins

# Away Wins for Each Team
away_wins = train[train['home_score'] < train['away_score']].groupby("away_team")['game_id'].nunique()
away_wins

# Total Wins/Total Games
total_wins = home_wins + away_wins
win_rate = total_wins/17

In [6]:
# Creating new df with all of our features
train_team = pd.DataFrame(win_rate)
train_team.index.name = "Team"
train_team.columns = ['win_rate']

In [7]:
# Time of Possession (Plays per Game)
plays_with_possession = train[~(train.play_type == "no_play")].groupby("posteam")['play_id'].nunique()
plays_per_game = plays_with_possession/17
train_team['plays/gm'] = plays_per_game

In [8]:
# Average Points Scored and Allowed (per Game) - ultimately chose to go with a per drive frequency for training

# 1) Sort by the game and play ids AND 2) group by each game and take the final row (final play) where we can see the final score
game_scores = train.sort_values(['game_id', 'play_id']).groupby("game_id").tail(1)[['game_id', 'home_team', 'away_team', 'home_score', 'away_score']]
game_scores['home_won'] = (game_scores['home_score'] > game_scores['away_score']).astype(int) # add an extra column for if the home team won; this is for later in the training process

# Computed average points scored and allowed based on home vs away
average_points_scored = (game_scores.groupby("home_team")['home_score'].sum() + game_scores.groupby("away_team")['away_score'].sum())/17
average_points_allowed = (game_scores.groupby("home_team")['away_score'].sum() + game_scores.groupby("away_team")['home_score'].sum())/17

In [9]:
# Team Points Forced (per Drive)
num_points_scored = (game_scores.groupby("home_team")['home_score'].sum() + game_scores.groupby("away_team")['away_score'].sum())
num_drives = train.groupby("posteam")['drive_play_id_ended'].nunique()

pts_per_drive = num_points_scored/num_drives

train_team['pts/drive'] = pts_per_drive

In [10]:
# Team Points Allowed (per Drive - limit)
num_points_allowed = (game_scores.groupby("home_team")['away_score'].sum() + game_scores.groupby("away_team")['home_score'].sum())
num_defensive_drives = train.groupby("defteam")['drive_play_id_ended'].nunique() 

points_allowed_per_drive = num_points_allowed/num_defensive_drives

train_team['pts_allowed/drive'] = points_allowed_per_drive

In [11]:
# Side Note (for proper filtering)
train.play_type.unique()

# Miscellaneous plays
train[train.play_type == "no_play"][['play_type', 'play_type_nfl']].play_type_nfl.unique()

array(['PENALTY', 'TIMEOUT', 'UNSPECIFIED', 'XP_KICK', 'KICK_OFF'],
      dtype=object)

<br><br><br><br><br><br><br>Offensive Statistics

In [12]:
# NOTE for statistics involving plays:
train.play_type_nfl.unique() # RUSH, PASS, SACK, INTERCEPTION, FUMBLE_RECOVERED_BY_OPPONENT only types of play possible for offense/defense

array(['GAME_START', 'KICK_OFF', 'RUSH', 'PASS', 'SACK', 'PUNT',
       'PENALTY', 'FIELD_GOAL', 'END_QUARTER', 'XP_KICK', 'TIMEOUT',
       'UNSPECIFIED', 'END_GAME', 'PAT2', 'INTERCEPTION', 'COMMENT',
       'FUMBLE_RECOVERED_BY_OPPONENT'], dtype=object)

In [13]:
# Average Yards (per Play - Pass + Rush + Sack)
yardage_plays = train[(train.play_type_nfl == 'RUSH') | (train.play_type_nfl == 'PASS') | (train.play_type_nfl == 'SACK')]
train_team['yds/play'] = yardage_plays.groupby("posteam")['yards_gained'].mean()

In [14]:
# Third Down Conversion Rate (per Attempt)
no_yes = train[(train.down == 3) & ~(train.play_type == "no_play")].groupby("posteam")['third_down_converted'].value_counts()

third_dn_conversion_rate = no_yes.unstack()[1.0]/(no_yes.unstack()[1.0] + no_yes.unstack()[0.0])

train_team["third_dn_conversion_rate"] = third_dn_conversion_rate

In [15]:
# Sacks Allowed (per Dropback - limit)
num_sacks_allowed = train[train.play_type_nfl == "SACK"].groupby("posteam")['play_id'].nunique()
num_dropbacks_offense = train[(train.play_type_nfl == "SACK") | (train.play_type_nfl == "PASS")].groupby("posteam")['play_id'].nunique()

pct_sacks_allowed = num_sacks_allowed/num_dropbacks_offense

train_team['sacks_allowed/dropback'] = pct_sacks_allowed

In [16]:
# Redzone Conversion Rate (per Drive)
num_redzone_tds = train[(train['drive_end_transition'] == "TOUCHDOWN") & (train['drive_inside20'] == 1)].groupby("posteam")['drive_play_id_ended'].nunique()
num_redzone_trips = train[(train['drive_inside20'] == 1)].groupby("posteam")['drive_play_id_ended'].nunique()

pct_redzone_td = num_redzone_tds/num_redzone_trips

train_team['pct_redzone_td'] = pct_redzone_td

In [17]:
# Turnovers Committed (per Drive)
num_committed_turnovers = train[(train.play_type_nfl == "INTERCEPTION") | (train.play_type_nfl == "FUMBLE_RECOVERED_BY_OPPONENT")].groupby("posteam")['play_id'].nunique()
num_offensive_drives = train.groupby("posteam")['drive_play_id_ended'].nunique() 

turnovers_committed_per_drive = num_committed_turnovers/num_offensive_drives

train_team["turnovers_committed/drive"] = turnovers_committed_per_drive

In [18]:
# Explosive Play Rate
num_explosive_plays = train[(train.yards_gained >= 20) & ((train.play_type_nfl == 'RUSH') | (train.play_type_nfl == 'PASS'))].groupby("posteam")['play_id'].nunique()
num_offensive_plays = train[((train.play_type_nfl == 'RUSH') | (train.play_type_nfl == 'PASS') | (train.play_type_nfl == 'INTERCEPTION') | (train.play_type_nfl == 'FUMBLE_RECOVERED_BY_OPPONENT') | (train.play_type_nfl == 'SACK'))].groupby("posteam")['play_id'].nunique()

pct_explosive_plays = num_explosive_plays/num_offensive_plays

train_team['pct_explosive_plays'] = pct_explosive_plays

In [19]:
# Negative Play Rate (limit)
num_negative_plays = train[(train.yards_gained < 0) & ((train.play_type_nfl == 'RUSH') | (train.play_type_nfl == 'PASS') | (train.play_type_nfl == 'INTERCEPTION') | (train.play_type_nfl == 'FUMBLE_RECOVERED_BY_OPPONENT') | (train.play_type_nfl == 'SACK'))].groupby("posteam")['play_id'].nunique()

pct_negative_plays = num_negative_plays/num_offensive_plays

train_team['pct_negative_plays'] = pct_negative_plays

<br><br><br><br><br><br><br>

Defensive Statistics

In [20]:
# Average Yards Allowed (per Play - Pass + Rush + Sack)
train_team['yds/play_allowed'] = yardage_plays.groupby("defteam")['yards_gained'].mean()

In [21]:
# Third Down Conversion Rate Allowed (per Attempt)
no_yes_def = train[(train.down == 3) & ~(train.play_type == "no_play")].groupby("defteam")['third_down_converted'].value_counts()

third_dn_conversion_rate_allowed = no_yes_def.unstack()[1.0]/(no_yes_def.unstack()[1.0] + no_yes_def.unstack()[0.0])

train_team["third_dn_conversion_rate_allowed"] = third_dn_conversion_rate_allowed

In [22]:
# Sacks Forced (per Dropback)
num_sacks = train[train.play_type_nfl == "SACK"].groupby("defteam")['play_id'].nunique()
num_dropbacks = train[(train.play_type_nfl == "SACK") | (train.play_type_nfl == "PASS")].groupby("defteam")['play_id'].nunique()

pct_sacks = num_sacks/num_dropbacks

train_team['sacks_forced/dropback'] = pct_sacks

In [23]:
# Redzone Stop Rate (per Drive)
num_redzone_tds_allowed = train[(train['drive_end_transition'] == "TOUCHDOWN") & (train['drive_inside20'] == 1)].groupby("defteam")['drive_play_id_ended'].nunique()
num_redzone_trips_allowed = train[(train['drive_inside20'] == 1)].groupby("defteam")['drive_play_id_ended'].nunique()

pct_redzone_td_allowed = num_redzone_tds_allowed/num_redzone_trips_allowed

train_team['pct_redzone_td_allowed'] = pct_redzone_td_allowed

In [24]:
# Turnovers Forced (per Drive)
num_forced_turnovers = train[(train.play_type_nfl == "INTERCEPTION") | (train.play_type_nfl == "FUMBLE_RECOVERED_BY_OPPONENT")].groupby("defteam")['play_id'].nunique()

turnovers_forced_per_drive = num_forced_turnovers/num_defensive_drives

train_team["turnovers_forced/drive"] = turnovers_forced_per_drive

In [25]:
# Explosive Play Rate Allowed
num_explosive_plays_allowed = train[(train.yards_gained >= 20) & ((train.play_type_nfl == 'RUSH') | (train.play_type_nfl == 'PASS'))].groupby("defteam")['play_id'].nunique()
num_defensive_plays = train[((train.play_type_nfl == 'RUSH') | (train.play_type_nfl == 'PASS') | (train.play_type_nfl == 'INTERCEPTION') | (train.play_type_nfl == 'FUMBLE_RECOVERED_BY_OPPONENT') | (train.play_type_nfl == 'SACK'))].groupby("defteam")['play_id'].nunique()

pct_explosive_plays_allowed = num_explosive_plays_allowed/num_defensive_plays

train_team['pct_explosive_plays_allowed'] = pct_explosive_plays_allowed

In [26]:
# Negative Play Rate Forced
num_negative_plays_forced = train[(train.yards_gained < 0) & ((train.play_type_nfl == 'RUSH') | (train.play_type_nfl == 'PASS') | (train.play_type_nfl == 'INTERCEPTION') | (train.play_type_nfl == 'FUMBLE_RECOVERED_BY_OPPONENT') | (train.play_type_nfl == 'SACK'))].groupby("defteam")['play_id'].nunique()

pct_negative_plays_forced = num_negative_plays_forced/num_defensive_plays

train_team['pct_negative_plays_forced'] = pct_negative_plays_forced

<br><br><br><br><br><br><br>

Special Teams Statistics

In [27]:
# Starting Field Position (per Drive)

# Group by Drive ID first and then group by Game ID if we have multiple of the same Drive IDs
team_yardstogo_id = train.sort_values('play_id').groupby(['drive_play_id_started', 'game_id'])[['posteam', 'yardline_100', 'play_id', 'drive_play_id_started']].head(1)
starting_field_pos = 100 - team_yardstogo_id.groupby('posteam').yardline_100.mean()

train_team['avg_starting_field_position'] = starting_field_pos

In [28]:
# FG Percentage
total_fgs_made = train[train.field_goal_result == "made"].groupby("posteam")['play_id'].nunique()
total_fgs_attempted = train[(train.field_goal_result == "made") | (train.field_goal_result == "missed") | (train.field_goal_result == "blocked")].groupby("posteam")['play_id'].nunique()

fg_pct = total_fgs_made/total_fgs_attempted

train_team['fg_pct'] = fg_pct

In [29]:
# Punts (per Drive - limit)
drives_ending_in_punts = train[train['drive_end_transition'] == "PUNT"].groupby("posteam")['drive_play_id_ended'].nunique()

punts_per_drive = drives_ending_in_punts/num_drives

train_team['punts/drive'] = punts_per_drive

<br><br><br><br><br><br><br>

### Data Modeling + Machine Learning

In [30]:
# Final Train Team DataFrame with all features
train_team = train_team.fillna(0) # to account for NYJ forcing N/A turnovers
train_team

,win_rate,plays/gm,pts/drive,pts_allowed/drive,yds/play,third_dn_conversion_rate,sacks_allowed/dropback,pct_redzone_td,turnovers_committed/drive,pct_explosive_plays,...,yds/play_allowed,third_dn_conversion_rate_allowed,sacks_forced/dropback,pct_redzone_td_allowed,turnovers_forced/drive,pct_explosive_plays_allowed,pct_negative_plays_forced,avg_starting_field_position,fg_pct,punts/drive
Team,,,,,,,,,,,,,,,,,,,,,
ARI,0.176471,68.117647,2.076023,2.757062,5.188324,0.413333,0.090352,0.540984,0.064327,0.058516,...,5.625347,0.428571,0.051601,0.632353,0.056497,0.068367,0.104082,51.786127,0.757576,0.345029
ATL,0.470588,65.529412,1.950276,2.265537,5.448508,0.330144,0.049242,0.620000,0.049724,0.060475,...,5.371373,0.395455,0.106145,0.549020,0.101695,0.067869,0.114528,47.375000,0.820513,0.348066
BAL,0.470588,63.000000,2.422857,2.235955,5.863967,0.406863,0.102975,0.482759,0.062857,0.075792,...,5.614884,0.378261,0.050505,0.516129,0.078652,0.078125,0.086458,49.376404,0.882353,0.302857
BUF,0.705882,66.823529,2.764368,2.085714,5.972923,0.447619,0.080808,0.661538,0.063218,0.078947,...,5.323718,0.413793,0.078091,0.595745,0.074286,0.059242,0.112559,45.542857,0.904762,0.281609
CAR,0.470588,63.470588,1.873494,2.331288,5.030030,0.355769,0.069034,0.533333,0.072289,0.057971,...,5.563000,0.471429,0.062500,0.551724,0.098160,0.062077,0.095937,48.424242,0.827586,0.331325
CHI,0.647059,68.529412,2.423077,2.331461,5.727190,0.423581,0.043478,0.589286,0.038462,0.073958,...,6.143856,0.402062,0.068493,0.561404,0.134831,0.077258,0.081610,48.540541,0.846154,0.324176
CIN,0.352941,66.470588,2.338983,2.843931,5.385948,0.432432,0.058252,0.666667,0.101695,0.058002,...,6.268151,0.434146,0.066667,0.634921,0.086705,0.080765,0.077577,48.882682,0.892857,0.367232
CLE,0.294118,66.176471,1.468421,1.953608,4.407517,0.334746,0.091562,0.526316,0.094737,0.042689,...,4.802789,0.361233,0.106122,0.591837,0.056701,0.061606,0.154015,45.087629,0.888889,0.473684
DAL,0.411765,71.823529,2.691429,2.920000,6.018987,0.411483,0.051325,0.569231,0.074286,0.067061,...,6.180328,0.465686,0.063869,0.666667,0.034286,0.077174,0.116304,49.926554,0.857143,0.234286


#### Training on Regular Season Matches:

In the training process, we will find the difference in features between the home team and the away team for each game in the NFL Regular Season. Each difference will have a corresponding label that indicates a win or loss (1/0) for the home team. During training, the model will learn how and to what extent features impact the result of a game.

In [31]:
"""
# Helper Function that flips the sign of statistics that teams want to limit
"""
def flip_limit(df):
    df['pts_allowed/drive'] *= -1
    df['sacks_allowed/dropback'] *= -1
    df['turnovers_committed/drive'] *= -1
    df['pct_negative_plays'] *= -1 
    df['yds/play_allowed'] *= -1 
    df['third_dn_conversion_rate_allowed'] *= -1 
    df['pct_redzone_td_allowed'] *= -1
    df['pct_explosive_plays_allowed'] *= -1 
    df['punts/drive'] *= -1

    return df

In [32]:
game_scores

,game_id,home_team,away_team,home_score,away_score,home_won
181,2025_01_ARI_NO,NO,ARI,13,20,0
372,2025_01_BAL_BUF,BUF,BAL,41,40,1
543,2025_01_CAR_JAX,JAX,CAR,26,10,1
713,2025_01_CIN_CLE,CLE,CIN,16,17,0
876,2025_01_DAL_PHI,PHI,DAL,24,20,1
...,...,...,...,...,...,...
45834,2025_18_NO_ATL,ATL,NO,19,17,1
45983,2025_18_NYJ_BUF,BUF,NYJ,35,8,1
46124,2025_18_SEA_SF,SF,SEA,3,13,0
46276,2025_18_TEN_JAX,JAX,TEN,41,7,1


In [33]:
"""
Creating the Training Data (per Game): 

1) X_train and y_train must have the same # of rows
2a) X_train should have # features columns
2b) y_train should have 1 column (what we are predicting) 

X_train = (# of NFL games, # of features); (272, 21); model learns from features
y_train = (did the Home Team win or lose --> 1 for yes, 0 for no); (272, 1); model learns the corresponding patterns of victory/loss from those features


We are iterating through all games, and subtracting the home_features - away_features to get a row of differences for each game. 
We also track the result of each game. The model then learns patterns about what features are important in deciding the winner/loser of a game.
"""
X_rows = []

for _, game in game_scores.iterrows():
    
    # For each game, find the home team and the away team
    home_team = game.home_team
    away_team = game.away_team

    """
    FOR EACH GAME:
    * we access the home team's statistics and subtract the away team's statistics
    * + : indicates home team advantage for that statistic; - : indicates away team advantage for that statistic
    """
    home_away_diff = train_team.loc[home_team] - train_team.loc[away_team]
    flip_limit(home_away_diff) # flips the sign of statistics that teams want to limit
    X_rows.append(home_away_diff) # add each game's difference statistics to a list

X_train = pd.DataFrame(X_rows) # convert the list of Series to a df
y_train = pd.Series(game_scores['home_won']) # our Series of win/loss

Our Training Dataframe and Series:

In [34]:
X_train

,win_rate,plays/gm,pts/drive,pts_allowed/drive,yds/play,third_dn_conversion_rate,sacks_allowed/dropback,pct_redzone_td,turnovers_committed/drive,pct_explosive_plays,...,yds/play_allowed,third_dn_conversion_rate_allowed,sacks_forced/dropback,pct_redzone_td_allowed,turnovers_forced/drive,pct_explosive_plays_allowed,pct_negative_plays_forced,avg_starting_field_position,fg_pct,punts/drive
0,0.176471,-1.235294,-0.307237,0.617397,-0.170433,-0.022222,0.007301,-0.096539,-0.005037,-0.018892,...,0.739441,0.087450,0.039492,0.100438,-0.000631,0.009544,0.019983,-4.037556,-0.043290,0.027110
1,0.235294,3.823529,0.341511,0.150241,0.108956,0.040756,0.022167,0.178780,-0.000361,0.003156,...,0.291166,-0.035532,0.027586,-0.079616,-0.004366,0.018883,0.026101,-3.833547,0.022409,0.021248
2,0.294118,4.823529,0.647783,0.524837,0.285077,0.034139,-0.002005,0.084314,-0.002179,0.010211,...,0.467245,0.069602,-0.010041,-0.035232,0.025496,0.008166,0.001315,-1.900433,0.054767,0.012176
3,-0.058824,-0.294118,-0.870562,0.890322,-0.978431,-0.097687,-0.033310,-0.140351,0.006958,-0.015313,...,1.465362,0.072913,0.039456,0.043084,-0.030004,0.019159,0.076438,-3.795053,-0.003968,-0.106453
4,0.235294,-7.705882,-0.500677,1.114444,-0.693987,-0.040055,-0.018957,0.135315,0.022263,-0.010770,...,1.155869,0.065686,0.013765,0.136054,0.049048,0.020451,-0.028069,-5.390241,-0.116402,-0.170339
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267,0.117647,-1.352941,0.181490,-0.125872,0.430617,-0.060968,0.033808,0.175556,0.019640,0.020851,...,-0.485467,-0.054333,0.015052,-0.017105,0.045829,-0.009045,-0.009536,-0.373571,0.106227,-0.030147
268,0.529412,1.588235,1.097701,0.724342,1.497595,0.099793,0.032473,0.214170,0.009004,0.035710,...,0.276838,-0.042901,0.026504,0.046046,0.074286,0.016309,0.007627,-4.326708,-0.060755,0.107280
269,-0.117647,2.294118,-0.097134,-0.678592,-0.374641,0.101096,0.007753,0.109142,-0.006218,-0.023005,...,-1.014317,-0.074359,-0.043816,-0.028658,-0.065103,-0.007344,-0.032556,4.089040,0.065603,0.023011
270,0.588235,3.823529,0.916757,0.894113,0.879931,0.075015,0.026353,0.041889,-0.023621,0.012065,...,0.759534,-0.012403,-0.029733,0.062166,0.084108,0.018472,-0.020789,-3.115535,0.082353,0.121529


In [35]:
y_train

181      0
372      1
543      1
713      0
876      1
        ..
45834    1
45983    1
46124    0
46276    1
46451    0
Name: home_won, Length: 272, dtype: int64

In [36]:
# Train the data on all Regular Season games' feature differences and the corresponding results
model = LogisticRegression()
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # scale data so a statistic like 'plays/gm' isn't overrepresented

model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


Below, we will calculate X_test (diff_stats) in the same format as X_train above for each Postseason game. We can then directly predict the probability of the result of each Postseason game. 

#### Predictions of Postseason Matches By Round:

We will be calculating the difference between the two playoff teams based on seeding. The higher seeded team will subtract the lower seeded team. Similar to our testing process, we will find the feature difference between the home and away team for each game, but for the NFL Postseason. Now, during testing, the model will evaluate each Postseason game based on patterns it found in Regular Season games + on how strong the playoff teams are analytically.

In [37]:
def predict_winner(home, away):
    home_stats = train_team.loc[[home]] # implicitly convert our home team's Series to a df to match dimensions for final evaluation
    away_stats = train_team.loc[away]
    
    # + : indicates home team advantage for that statistic; - : indicates away team advantage for that statistic
    diff_stats = home_stats - away_stats
    
    # Flipping sign of statistics that teams want to limit
    flip_limit(diff_stats)

    # Results - evaluate the chances of winning the game based on Regular Season data + how Postseason teams' compare against each other
    home_chance = round(model.predict_proba(diff_stats)[0][1] * 100, 2)
    away_chance = round(100 - home_chance, 2)

    predicted_winner = None
    
    if home_chance >= away_chance:
        predicted_winner = home
    else:
        predicted_winner = away

    print(f"{home} has a {home_chance}% chance to win and {away} has a {away_chance}% chance to win.")
    print(f"Predicted Winner: {predicted_winner}")

##### <u>Wild Card</u>

NFC:

In [38]:
# 2) CHI vs 7) GB

In [39]:
# We will perform this calculation by hand (simulating predict_winner()) to have a better understanding of each functioning component:

home_stats = train_team.loc['CHI']
away_stats = train_team.loc['GB']

diff_stats = home_stats - away_stats # calculate the difference in statistics between the two teams
flip_limit(diff_stats) # flipping sign of statistics that teams want to limit

# + : indicates home team advantage for that statistic; - : indicates away team advantage for that statistic
diff_stats 

win_rate                            0.117647
plays/gm                            3.470588
pts/drive                           0.024304
pts_allowed/drive                  -0.038467
yds/play                            0.090101
third_dn_conversion_rate           -0.054888
sacks_allowed/dropback              0.017318
pct_redzone_td                      0.005952
turnovers_committed/drive           0.004483
pct_explosive_plays                 0.004499
pct_negative_plays                  0.004991
yds/play_allowed                   -1.109668
third_dn_conversion_rate_allowed   -0.009368
sacks_forced/dropback               0.002317
pct_redzone_td_allowed              0.034750
turnovers_forced/drive              0.083876
pct_explosive_plays_allowed        -0.025733
pct_negative_plays_forced          -0.005666
avg_starting_field_position         1.941738
fg_pct                             -0.002331
punts/drive                        -0.029697
dtype: float64

In [40]:
# Train the data on all Regular Season games' feature differences and the corresponding results
model = LogisticRegression(fit_intercept = False)
model.fit(X_train, y_train)

# Results - evaluate the chances of winning the game based on Regular Season data + how Postseason teams' compare against each other
chi_chance = round(model.predict_proba(diff_stats.to_frame().T)[0][1] * 100, 2)
gb_chance = round(100 - chi_chance, 2)

predicted_winner = None

if chi_chance >= gb_chance:
    predicted_winner = "CHI"
else:
    predicted_winner = "GB"

print(f"CHI has a {chi_chance}% chance to win and GB has a {gb_chance}% chance to win.")
print(f"Predicted Winner: {predicted_winner}")

CHI has a 55.72% chance to win and GB has a 44.28% chance to win.
Predicted Winner: CHI


<br><br>

In [41]:
# 3) PHI vs 6) SF
predict_winner('PHI', 'SF')

PHI has a 50.07% chance to win and SF has a 49.93% chance to win.
Predicted Winner: PHI


In [42]:
# 4) CAR vs 5) LA
predict_winner('CAR', 'LA')

CAR has a 17.4% chance to win and LA has a 82.6% chance to win.
Predicted Winner: LA


AFC:

In [43]:
# 2) NE vs 7) LAC
predict_winner('NE', 'LAC')

NE has a 70.47% chance to win and LAC has a 29.53% chance to win.
Predicted Winner: NE


In [44]:
# 3) JAX vs 6) BUF
predict_winner('JAX', 'BUF')

JAX has a 52.41% chance to win and BUF has a 47.59% chance to win.
Predicted Winner: JAX


In [45]:
# 4) PIT vs 5) HOU
predict_winner('PIT', 'HOU')

PIT has a 33.0% chance to win and HOU has a 67.0% chance to win.
Predicted Winner: HOU


##### <u>Divisional</u>

NFC:

In [46]:
# 1) SEA vs 6) SF
predict_winner('SEA', 'SF')

SEA has a 71.59% chance to win and SF has a 28.41% chance to win.
Predicted Winner: SEA


In [47]:
# 2) CHI vs 5) LA
predict_winner('CHI', 'LA')

CHI has a 31.7% chance to win and LA has a 68.3% chance to win.
Predicted Winner: LA


AFC:

In [48]:
# 1) DEN vs 6) BUF
predict_winner('DEN', 'BUF')

DEN has a 55.19% chance to win and BUF has a 44.81% chance to win.
Predicted Winner: DEN


In [49]:
# 2) NE vs 5) HOU
predict_winner('NE', 'HOU')

NE has a 61.31% chance to win and HOU has a 38.69% chance to win.
Predicted Winner: NE


##### <u>Conference Championship</u>

In [50]:
# 1) SEA vs 5) LA
predict_winner('SEA', 'LA')

SEA has a 62.24% chance to win and LA has a 37.76% chance to win.
Predicted Winner: SEA


In [51]:
# 1) DEN vs 2) NE
predict_winner('DEN', 'NE')

DEN has a 45.55% chance to win and NE has a 54.45% chance to win.
Predicted Winner: NE


##### <u>Super Bowl</u>

We will be calculating the difference between the two playoff teams based on conference. The team representing the NFC will subtract the team representing the AFC.

In [52]:
# SEA vs NE
predict_winner('SEA', 'NE')

SEA has a 56.73% chance to win and NE has a 43.27% chance to win.
Predicted Winner: SEA
